In [4]:
import cv2
import os
import numpy as np
from skimage.metrics import structural_similarity as ssim
import img2pdf
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from time import sleep



In [5]:
from pathlib import Path

VIDEO_NAME = "Twenty One Pilots - The Run And Go"
VIDEO_NAME = "Ludovico Einaudi - Experience"
VIDEO_PATH = Path("./videos") / f"{VIDEO_NAME}.mp4"
OUTPUT_DIR = "sheet_frames"
PDF_OUTPUT = f"{VIDEO_NAME}.pdf"

FRAME_INTERVAL = 1  # process every 10 frames
DIFF_THRESHOLD = 0.95  # SSIM threshold

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
import cv2
import numpy as np

def plot_append(image, variable, plot_width_height=40, position="right", color=(100, 100, 255)):

    if position in ("right", "left"):
        plot_height = frame.shape[0]
        plot_width = plot_width_height
        plot_img = np.ones((plot_height, plot_width, 3), dtype=np.uint8) * 255
    elif position in ("top", "bottom"):
        plot_height = plot_width_height 
        plot_width = frame.shape[1]
        plot_img = np.ones((plot_height, plot_width, 3), dtype=np.uint8) * 255

    # Normalize for plotting
    norm_variable = (variable - variable.min()) / (np.ptp(variable) + 1e-6)
    pos = int(norm_variable[y] * (plot_width_height - 1))
    for y in range(len(norm_variable)):
        if position == "right":
            plot_img[y, :pos, :] = color
        elif position == "left":
            plot_img[y, -pos:, :] = color
        elif position == "top":
            plot_img[:pos, y, :] = color
        elif position == "bottom":
            plot_img[-pos:, y, :] = color
    
    # Instead of overlaying, concatenate the plot image 
    # Resize plot_img height to match frame if needed
    if position in ("right", "left"):
        if plot_img.shape[0] != image.shape[0]:
            plot_img = cv2.resize(plot_img, (plot_img.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
    elif position in ("top", "bottom"):
        if plot_img.shape[1] != image.shape[1]:
            plot_img = cv2.resize(plot_img, (image.shape[1], plot_img.shape[0]), interpolation=cv2.INTER_NEAREST)

    if position == "right":
        image = np.concatenate((image, plot_img), axis=1)
    elif position == "left":
        image = np.concatenate((plot_img, image), axis=1)
    elif position == "top":
        image = np.concatenate((plot_img, image), axis=0)
    elif position == "bottom":
        image = np.concatenate((image, plot_img), axis=0)
    
    return image

def get_bar_position(frame, ymax, draw=False):

    pentagram_detection_x = 30
    cropped_left = frame[:, 0:pentagram_detection_x]  

    # Convert the cropped left region to HSV color space for color-based processing
    # HSV stands for Hue, Saturation, and Value:
    # - Hue: the color type (angle from 0 to 179 in OpenCV)
    # - Saturation: intensity or purity of the color (0 = gray, 255 = pure color)
    # - Value: brightness of the color (0 = black, 255 = brightest)
    # HSV is often used for color segmentation because it separates color (hue) from intensity (value)
    hsv = cv2.cvtColor(cropped_left, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    # Sum horizontally to find the limit between the pentagram and the piano area
    avg_s = np.sum(s, axis=1)
    avg_v = np.sum(v, axis=1)
    # Find the first row where saturation drops below a threshold
    # This helps to find the boundary where the pentagram ends and the piano starts
    pentagram_end = int(frame.shape[0]*0.35)#np.argmax(avg_s < 30)

    # Crop the top part of the frame to focus on the bar
    cropped_top = frame[0:int(pentagram_end), :]  

    # Convert to HSV to filter by brightness and darkness
    hsv = cv2.cvtColor(cropped_top, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    # Mask out very bright (whitish) and very dark (blackish) areas
    # Keep only mid-brightness (colored) pixels
    # You may need to tune these thresholds for your images
    bright_thresh = 210  # above this is "whitish"
    dark_thresh = 40     # below this is "blackish"
    sat_thresh = 30
    mask = (v > dark_thresh) & (v < bright_thresh)
    dark_ratio = np.sum((v < dark_thresh) & (s < sat_thresh)) / mask.size  # proportion of dark pixels
    bright_ratio = np.sum((v > bright_thresh) & (s < sat_thresh)) / mask.size  # proportion of bright pixels

    # Optionally, also require some saturation to avoid gray
    mask = mask & (s > sat_thresh)

    # Mask to keep only mid-tones (the bar)
    #mask = cv2.inRange(gray, lower_thresh, upper_thresh)
    # Compute the proportion of mask pixels to the total number of pixels in the cropped area
    mask_ratio = np.sum(mask > 0) / mask.size

    # Sum of pixels along vertical axis (per column)
    # Compute sliding moving average along the column_sum
    window_size = 15  # You can adjust the window size as needed
    column_sum = np.sum(mask, axis=0)
    column_sum = np.convolve(column_sum, np.ones(window_size)/window_size, mode='same')

    # Find the column with the max sum
    bar_x = np.argmax(column_sum)

    if draw:
        vis = frame.copy()

        vis = plot_append(vis, avg_s, plot_width_height=30, position="right", color=(255, 0, 0))
        vis = plot_append(vis, avg_v, plot_width_height=30, position="right", color=(0, 255, 0))
        vis = plot_append(vis, column_sum, plot_width_height=30, position="top", color=(0, 0, 255))
        #vis[0:plot_height, 0:plot_width] = plot_img

        cv2.line(vis, (bar_x, 0), (bar_x, ymax), (0, 0, 255), 1)
        cv2.line(vis, (pentagram_detection_x, 0), (pentagram_detection_x, vis.shape[0]), (0, 0, 255), 1)
        cv2.line(vis, (0, pentagram_end+30), (vis.shape[1], pentagram_end+30), (0, 0, 255), 1)
        cv2.putText(vis, f"Mask ratio: {mask_ratio*100:.3f}%", (10, vis.shape[0]-50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        cv2.putText(vis, f"Dark ratio: {dark_ratio*100:.3f}%", (10, vis.shape[0]-30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        cv2.putText(vis, f"Bright ratio: {bright_ratio*100:.3f}%", (10, vis.shape[0]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        return bar_x, vis

    return bar_x

cap = cv2.VideoCapture(VIDEO_PATH)
ymax = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) * 0.35)  # crop top third
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    bar_x, vis = get_bar_position(frame, ymax, draw=True)

    cv2.imshow("Bar Detection", vis)
    if cv2.waitKey(30) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 

In [ ]:
def crop_sheet_area(frame):
    height = frame.shape[0]
    cropped = frame[0:int(height*0.35), :]  # crop top third
    return cropped

def is_different(prev, curr):
    prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
    curr_gray = cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY)
    score, _ = ssim(prev_gray, curr_gray, full=True)
    return score < DIFF_THRESHOLD


def show_image(img):
    # Convert BGR (OpenCV default) to RGB for matplotlib
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img_rgb)
    plt.axis("off")
    plt.show()

def has_pentagram(image, debug=False):
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Threshold to binary
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Morphological filtering to isolate horizontal lines
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (20, 1))
    detected_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)

    # Find contours of these lines
    contours, _ = cv2.findContours(detected_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Count the number of horizontal line groups
    line_y_positions = [cv2.boundingRect(cnt)[1] for cnt in contours]
    line_y_positions.sort()

    # Group close Y positions (to allow small vertical jitter)
    #print(np.diff(line_y_positions))
    grouped_lines = []
    for y in line_y_positions:
        if not grouped_lines or abs(grouped_lines[-1][-1] - y) > 20:
            grouped_lines.append([y])
        else:
            grouped_lines[-1].append(y)

    # Look for groups with ~5 lines (allow 4-6 for flexibility)
    pentagram_count = sum(1 for group in grouped_lines if 4 <= len(group) <= 6)

    if debug:
        print(f"Detected {pentagram_count} pentagrams")
        show_image(binary)
        show_image(detected_lines)


    return pentagram_count >= 1


def detect_blue_bar_x(image, debug=False):
    # Convert to HSV for better color segmentation
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Define blue color range in HSV
    lower_blue = np.array([110, 200, 200])
    upper_blue = np.array([130, 255, 255])

    # Threshold to get blue areas
    mask = cv2.inRange(hsv, lower_blue, upper_blue)

    # Morphological filter to strengthen vertical bars
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 30))
    vertical_mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, vertical_kernel, iterations=1)

    # Find contours in the mask
    contours, _ = cv2.findContours(vertical_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if debug:
        debug_img = image.copy()
        show_image(mask)
        show_image(vertical_mask)
        show_image(debug_img)
        show_image(debug_img)


    if not contours:
        return None

    # Choose the leftmost blue vertical contour (smallest x)
    contour = min(contours, key=lambda cnt: cv2.boundingRect(cnt)[0])
    x, y, w, h = cv2.boundingRect(contour)
    return x + w // 2  # Return center X coordinate

In [ ]:
import cv2
from IPython.display import display, Image
cap = cv2.VideoCapture(VIDEO_PATH)

display_handle=display(None, display_id=True)
try:
    while True:
        break
        _, frame = cap.read()
        _, frame = cv2.imencode('.jpeg', frame)
        display_handle.update(Image(data=frame.tobytes()))
finally:
    cap.release()
    display_handle.update(None)

None

In [ ]:

cap = cv2.VideoCapture(VIDEO_PATH)


fps = cap.get(cv2.CAP_PROP_FPS)
total_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
length = total_frame_count/fps

#pbar = tqdm(total = total_frame_count)

prev_pos = None
curr_pos = None
while(True):
    ret, frame = cap.read()
    if not ret:
        break

    #show_image(frame)
    curr_pos = detect_blue_bar_x(frame, False)
    if curr_pos and not prev_pos:
        pass
    elif not curr_pos and prev_pos:
        detect_blue_bar_x(frame, True)
    elif not curr_pos and not prev_pos:
        pass
    elif curr_pos < prev_pos:
        print("New frame!")
        print(curr_pos, prev_pos)
        detect_blue_bar_x(frame, True)
    if curr_pos:
        prev_pos = curr_pos
    #sleep(0.05)
    #pent.append(has_pentagram(frame, False))
    #pbar.update(fps*5)


In [ ]:
from collections import deque
from PIL import Image as PILImage
import numpy as np

def strong_black_ratio(image, threshold=40, gray_tolerance=15):
    """
    Returns the ratio of strong black pixels in the image.
    """
    
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    black_mask = (gray < threshold) & (hsv[...,1] < 30)
    return np.sum(black_mask) / black_mask.size

def compute_ssim(img1, img2):
    img1_gray = cv2.cvtColor(img1, cv2.COLOR_BGR2HSV)
    img2_gray = cv2.cvtColor(img2, cv2.COLOR_BGR2HSV)
    score, _ = ssim(img1_gray, img2_gray, full=True, multichannel=True)
    return score
    
def compute_absdiff(img1, img2):
    img1_gray = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    img2_gray = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    diff = cv2.absdiff(img1_gray, img2_gray)
    return np.mean(diff)
   

total_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

offset = int(cap.get(cv2.CAP_PROP_FPS))
N = 1

pbar = tqdm(total=(total_frame_count - offset) // N)

cap = cv2.VideoCapture(VIDEO_PATH)

idx = []
sim = []
ratio = []
frame_queue = deque(maxlen=offset + 1)
frame_idx = 0

state = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    cropped = crop_sheet_area(frame)
    

    # if state == 0:
    #     if detect_blue_bar_x(cropped, False) is not None:
    #         state = 1
    #     else:
    #         continue
    state = 1

    frame_queue.append(cropped)
    if frame_idx >= offset and frame_idx % N == 0:
        curr_crop = frame_queue[-1]
        prev_crop = frame_queue[0]

        abs_diff = compute_absdiff(curr_crop, prev_crop)
        print(abs_diff)
        if state == 1 and abs_diff > 3:
            if detect_blue_bar_x(cropped, False) is not None:
                out_path = os.path.join(OUTPUT_DIR, f"sheet_{frame_idx:05d}.png")
                cv2.imwrite(out_path, prev_crop)
            state = 2
        elif state == 2 and abs_diff < 3:
            state = 1
        # if abs_diff > 3:
        #     save_next = True
        # elif 'save_next' in locals() and save_next:
        #     out_path = os.path.join(OUTPUT_DIR, f"sheet_{frame_idx:03d}.png")
        #     cv2.imwrite(out_path, curr_crop)
        #     save_next = False
            
        idx.append(frame_idx)
        sim.append(abs_diff)
        ratio.append(strong_black_ratio(curr_crop))
    
    pbar.update(1)
    frame_idx += 1

cap.release()
pbar.close()

0it [00:00, ?it/s]

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0


In [ ]:

# Convert to PDF
# A4 size in pixels at 300 DPI
A4_WIDTH_PX = 2480
A4_HEIGHT_PX = 3508
TOP_MARGIN = 150

# Load all images as PIL Images and resize to fit A4 width
image_paths = [os.path.join(OUTPUT_DIR, img) for img in sorted(os.listdir(OUTPUT_DIR))]
sheet_imgs = [PILImage.open(p).convert("RGB") for p in image_paths]

# Optionally resize images to fit A4 width
resized_imgs = []
for img in sheet_imgs:
    if img.width != A4_WIDTH_PX:
        new_height = int(img.height * (A4_WIDTH_PX / img.width))
        img = img.resize((A4_WIDTH_PX, new_height), PILImage.LANCZOS)
    resized_imgs.append(img)


def process_page(page: PILImage):
    # Convert to greyscale and enhance white background
    gray = page.convert("L")
    # Enhance white background: threshold to make near-white pixels pure white
    arr = np.array(gray)
    # Set a threshold, e.g., 240 out of 255
    arr[arr > 200] = 255
    # Convert back to "L" mode image
    enhanced = PILImage.fromarray(arr, mode="L")
    # Convert back to RGB for PDF saving
    return enhanced.convert("RGB")

pages = []
current_y = 0
current_page = PILImage.new("RGB", (A4_WIDTH_PX, A4_HEIGHT_PX), "white")

current_y = TOP_MARGIN  # Start pasting images below the top margin

for img in resized_imgs:
    if current_y + img.height > A4_HEIGHT_PX:
        
        pages.append(process_page(current_page))
        current_page = PILImage.new("RGB", (A4_WIDTH_PX, A4_HEIGHT_PX), "white")
        current_y = TOP_MARGIN
    current_page.paste(img, (0, current_y))
    current_y += img.height

# Add the last page if it has content
if current_y > 0:
    pages.append(process_page(current_page))

# Save as PDF
pages[0].save(PDF_OUTPUT, save_all=True, append_images=pages[1:], resolution=300)

print(f"✅ Done. PDF saved as: {PDF_OUTPUT}")



✅ Done. PDF saved as: Twenty One Pilots - The Run And Go.pdf


In [ ]:
from scipy.signal import find_peaks

plt.plot(idx, ratio)
plt.xlabel("Frame Index")
plt.ylabel("SSIM Similarity")
plt.title("SSIM Similarity Over Frames")
#plt.xlim([200, 400])
plt.yscale("log")
plt.show()

peaks, _ = find_peaks(sim, height=np.mean(sim))
plt.plot(idx, sim)
plt.plot(np.array(idx)[peaks], np.array(sim)[peaks], "x", label="Peaks")
plt.xlabel("Frame Index")
plt.ylabel("SSIM Similarity")
plt.title("SSIM Similarity Over Frames with Peaks")
plt.yscale("log")
plt.legend()
plt.show()

In [ ]:
import cv2
import numpy as np

def is_sharp_or_blurred(img, show_metrics=False):
    """
    Simple function to classify if an image has sharp blacks or is blurred/grey
    
    Returns:
    - 'sharp': Image has good contrast with strong blacks
    - 'blurred': Image is blurred with mostly grey tones
    """
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Method 1: Laplacian variance (sharpness measure)
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    laplacian_var = laplacian.var()
    
    # Method 2: Standard deviation (contrast measure)  
    std_dev = np.std(gray)
    
    # Method 3: Black/white pixel analysis
    black_pixels = np.sum(gray < 50)  # Very dark pixels
    white_pixels = np.sum(gray > 200)  # Very bright pixels
    total_pixels = gray.shape[0] * gray.shape[1]
    
    binary_ratio = (black_pixels + white_pixels) / total_pixels
    
    if show_metrics:
        print(f"Laplacian Variance: {laplacian_var:.2f}")
        print(f"Standard Deviation: {std_dev:.2f}")
        print(f"Binary Ratio (black+white): {binary_ratio:.3f}")
    
    # Decision logic (thresholds may need adjustment for your specific images)
    sharp_indicators = 0
    
    if laplacian_var > 500:  # Good sharpness
        sharp_indicators += 1
    if std_dev > 60:  # Good contrast
        sharp_indicators += 1  
    if binary_ratio > 0.4:  # Mostly black or white pixels
        sharp_indicators += 1
        
    # Classify based on how many indicators suggest sharpness
    # if sharp_indicators >= 2:
    #     return 'sharp'
    # else:
    #     return 'blurred'
    return laplacian_var, std_dev, binary_ratio


frame_idx = 0
saved_idx = 0
prev_crop = None

cap = cv2.VideoCapture(VIDEO_PATH)


total_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
pbar = tqdm(total=total_frame_count)


while True:
    ret, frame = cap.read()
    pbar.update(1)
    if not ret:
        break
    cropped = crop_sheet_area(frame)
    if detect_blue_bar_x(cropped, False) is not None:
        break

a=[]
b=[]
c=[]

while True:
    ret, frame = cap.read()
    pbar.update(1)
    if not ret:
        break
    cropped = crop_sheet_area(frame)

    laplacian_var, std_dev, binary_ratio = is_sharp_or_blurred(cropped, False)
    a.append(laplacian_var)
    b.append(std_dev)
    c.append(binary_ratio)  
cap.release()


In [ ]:
plt.plot(a, label='Laplacian Variance')
plt.show()
plt.plot(b, label='Standard Deviation')
plt.show()
plt.plot(c, label='Binary Ratio')
plt.show()
plt.xlabel("Frame Index")
plt.ylabel("Metric Value")

In [ ]:
# Convert to PDF
with open(PDF_OUTPUT, "wb") as f:
    images = [os.path.join(OUTPUT_DIR, img) for img in sorted(os.listdir(OUTPUT_DIR))]
    f.write(img2pdf.convert(images))

print(f"✅ Done. PDF saved as: {PDF_OUTPUT}")